# 02 — Feature Research

Analyse feature quality: information coefficients, feature importance,
and autocorrelation.  Claude interprets the feature landscape and
suggests engineering improvements.

**Cost per full run**: ~£0.02 (one Claude API call at ~1k tokens).

Runs without an API key — all charts render; interpretations show unavailable.

### Path setup

`sys.path.insert(0, "..")` makes `src` importable from `notebooks/`.

In [ ]:
import sys
sys.path.insert(0, "..")

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.utils.config import load_config
from src.services.data_service import DataService
from src.features.pipeline import FeaturePipeline
from src.notebooks.formatters import format_feature_stats, display_interpretation
from src.notebooks.research_log import ResearchLog

In [ ]:
config = load_config()
data_service = DataService(config)
pipeline = FeaturePipeline(config)
research_log = ResearchLog(config)

tickers = config["universe"]["tickers"]
print(f"Universe: {tickers}")

## Section 1 — Generate Features

In [ ]:
# Load OHLCV for the first ticker and generate features.
ticker = tickers[0]
ohlcv = data_service.get_price_data(ticker)
features_df = pipeline.generate(ohlcv)

feature_names = pipeline.get_feature_names()
print(f"Generated {len(feature_names)} features for {ticker}")
print(f"Shape: {features_df.shape}")
features_df[feature_names].describe().round(4)

## Section 2 — Feature-Target IC Analysis

In [ ]:
# Compute Information Coefficient (rank correlation with forward returns).
target_col = config["models"].get("target_column", "forward_return_5d")

if target_col not in features_df.columns:
    # Create a simple forward return target.
    features_df[target_col] = features_df["Close"].pct_change(5).shift(-5)
    print(f"Created target: {target_col}")

# Rank IC per feature.
ic_values = {}
for feat in feature_names:
    valid = features_df[[feat, target_col]].dropna()
    if len(valid) > 30:
        ic = valid[feat].corr(valid[target_col], method="spearman")
        ic_values[feat] = round(ic, 4)

ic_series = pd.Series(ic_values).sort_values(key=abs, ascending=False)
print(f"\nTop features by |IC|:")
ic_series.head(10)

In [ ]:
# IC bar chart.
fig = px.bar(
    x=ic_series.index[:15],
    y=ic_series.values[:15],
    title=f"Feature Information Coefficients ({ticker})",
    labels={"x": "Feature", "y": "Spearman IC"},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## Section 3 — Feature Importance (from trained model)

In [ ]:
# Try to load a pre-trained model for feature importance.
try:
    from src.services.model_service import ModelService

    model_service = ModelService(config)
    importance = model_service.get_feature_importance()
    if importance is not None and len(importance) > 0:
        imp_series = pd.Series(importance).sort_values(ascending=False).head(15)
        fig = px.bar(
            x=imp_series.index,
            y=imp_series.values,
            title="Model Feature Importance (Top 15)",
            labels={"x": "Feature", "y": "Importance"},
        )
        fig.update_layout(xaxis_tickangle=-45)
        fig.show()
    else:
        print("No trained model available — skipping feature importance chart.")
        print("Run model training first: py -3 -m scripts.run_backtest")
except Exception as e:
    print(f"Feature importance unavailable: {e}")

## Section 4 — Claude Interpretation

In [ ]:
# Prepare structured data for Claude.
feature_stats = format_feature_stats(features_df)
feature_stats["ic_values"] = ic_values
feature_stats["ticker"] = ticker

# Prior context from earlier notebooks.
prior_entries = research_log.load_recent(3)
context = {
    "prior_research": [
        {"notebook": e["notebook"], "task": e["task"], "summary": e["interpretation"].get("summary", "")}
        for e in prior_entries
    ]
} if prior_entries else None

print(f"Data prepared: {feature_stats['feature_count']} features, {len(ic_values)} IC values")

In [ ]:
try:
    from src.notebooks.claude_interpreter import QuantInterpreter

    interpreter = QuantInterpreter(config)
    interpretation = interpreter.interpret("feature_analysis", feature_stats, context=context)
except (EnvironmentError, ImportError) as e:
    print(f"Claude interpretation unavailable: {e}")
    interpretation = {
        "summary": "Interpretation unavailable (no API key set)",
        "observations": [],
        "warnings": ["Set ANTHROPIC_API_KEY to enable interpretations"],
        "suggestions": [],
        "confidence": "low",
    }

display_interpretation(interpretation)

In [ ]:
research_log.log_entry(
    notebook="02_feature_research",
    task="feature_analysis",
    data_summary=feature_stats,
    interpretation=interpretation,
)
print("Entry logged.")